# Behavioural Reaction-Time Analysis: Post-Error Slowing

# Aim
This notebook demonstrates a basic behavioural reaction-time analysis pipeline...

## Create sample dataset

In [1]:
import pandas as pd
data = {
    "participant": [
        1, 1, 1, 1, 1, 1, 1, 1,
        2, 2, 2, 2, 2, 2, 2, 2,
        3, 3, 3, 3, 3, 3, 3, 3
    ],
    "trial": [
        1, 2, 3, 4, 5, 6, 7, 8,
        1, 2, 3, 4, 5, 6, 7, 8,
        1, 2, 3, 4, 5, 6, 7, 8
    ],
    "accuracy": [
        1, 0, 1, 1, 0, 1, 1, 0,
        1, 1, 0, 1, 0, 1, 1, 0,
        0, 1, 1, 0, 1, 1, 0, 1
    ],
    "RT": [
        500, 450, 700, 120, 800, 850, 3500, 600,
        520, 540, 480, 760, 150, 810, 900, 4000,
        600, 780, 620, 500, 830, 650, 110, 870
    ]
}
df = pd.DataFrame(data)
display(df)

,participant,trial,accuracy,RT
0,1,1,1,500
1,1,2,0,450
2,1,3,1,700
3,1,4,1,120
4,1,5,0,800
5,1,6,1,850
6,1,7,1,3500
7,1,8,0,600
8,2,1,1,520
9,2,2,1,540


## Clean reaction times
Reaction-time data can contain unrealistic values. Here, RTs below 200 ms and above 1000 ms are excluded before analysis.

In [2]:
clean_df = df[(df["RT"]> 200) & (df["RT"] < 1000)]
display(clean_df)

clean_df["previous_accuracy"] = clean_df.groupby("participant")["accuracy"].shift(1)

,participant,trial,accuracy,RT
0,1,1,1,500
1,1,2,0,450
2,1,3,1,700
4,1,5,0,800
5,1,6,1,850
7,1,8,0,600
8,2,1,1,520
9,2,2,1,540
10,2,3,0,480
11,2,4,1,760


## Create previous-trial labels
Each trial is labelled according to whether the previous valid trial was correct or an error. This is calculated separately for each participant.

In [3]:
clean_df["previous_trial"] = clean_df["previous_accuracy"].map({ 1 : "post_correct", 0: "post_error"})
display(clean_df)

df_clean = clean_df.dropna()
display(df_clean)

mean_post_error_results_ovr = df_clean.groupby("previous_trial")["RT"].mean()

display(mean_post_error_results_ovr )

post_correct_rt_ovr = mean_post_error_results_ovr["post_correct"]
post_error_rt_ovr = mean_post_error_results_ovr["post_error"]

,participant,trial,accuracy,RT,previous_accuracy,previous_trial
0,1,1,1,500,NaN,NaN
1,1,2,0,450,1.0,post_correct
2,1,3,1,700,0.0,post_error
4,1,5,0,800,1.0,post_correct
5,1,6,1,850,0.0,post_error
7,1,8,0,600,1.0,post_correct
8,2,1,1,520,NaN,NaN
9,2,2,1,540,1.0,post_correct
10,2,3,0,480,1.0,post_correct
11,2,4,1,760,0.0,post_error


,participant,trial,accuracy,RT,previous_accuracy,previous_trial
1,1,2,0,450,1.0,post_correct
2,1,3,1,700,0.0,post_error
4,1,5,0,800,1.0,post_correct
5,1,6,1,850,0.0,post_error
7,1,8,0,600,1.0,post_correct
9,2,2,1,540,1.0,post_correct
10,2,3,0,480,1.0,post_correct
11,2,4,1,760,0.0,post_error
13,2,6,1,810,1.0,post_correct
14,2,7,1,900,1.0,post_correct


previous_trial
post_correct    656.363636
post_error      784.000000
Name: RT, dtype: float64

## Overall post-error slowing

In [4]:
post_error_slowing_ovr = post_error_rt_ovr - post_correct_rt_ovr
display(post_error_slowing_ovr)

np.float64(127.63636363636363)

## Participant-level mean RT

In [5]:
mean_post_error_results_parti = df_clean.groupby(["participant","previous_trial"])["RT"].mean()
display(mean_post_error_results_parti)

post_error_results = mean_post_error_results_parti.unstack()
display(post_error_results)

post_correct_rt_parti = post_error_results["post_correct"]
post_error_rt_parti = post_error_results["post_error"]

participant  previous_trial
1            post_correct      616.666667
             post_error        775.000000
2            post_correct      682.500000
             post_error        760.000000
3            post_correct      660.000000
             post_error        805.000000
Name: RT, dtype: float64

previous_trial,post_correct,post_error
participant,,
1,616.666667,775.0
2,682.500000,760.0
3,660.000000,805.0


## Participant-level post-error slowing

In [6]:
post_error_results["post_error_slowing"] = (post_error_results["post_error"] - post_error_results["post_correct"])
display(post_error_results)

mean_post_error_slowing = post_error_results["post_error_slowing"].mean()

# Print the final mean post-error slowing value
print("Mean post-error slowing across participants:", mean_post_error_slowing, "ms")

previous_trial,post_correct,post_error,post_error_slowing
participant,,,
1,616.666667,775.0,158.333333
2,682.500000,760.0,77.500000
3,660.000000,805.0,145.000000


Mean post-error slowing across participants: 126.94444444444446 ms


## Interpretation

This analysis compared reaction times after correct trials and after error trials in a simulated behavioural dataset. Positive post-error slowing values indicate that participants responded more slowly following errors. The analysis was conducted both overall and at participant level, with participant-level scores providing a cleaner summary for group-level interpretation.

Because this is a small simulated dataset, the results are for demonstration rather than statistical inference.